# AC Stark Shift Data Analysis (Runnable)

This notebook is a runnable, repository-local version of the original `data_analysis.ipynb`.

## What was fixed
- Removed lab-only dependencies (`qm`, `qutip`, `pyvisa`, custom `opx_config_*`, etc.).
- Added robust path resolution so it runs from any working directory.
- Uses only standard scientific Python deps from `requirements.txt` (`numpy`, `matplotlib`, optional `scipy`).
- Preserves the original fitting logic (`delta_Q_freq` with scale-factor fitting, including Q2 filtered refit).

## Inputs
- Embedded AC-frequency sweep arrays (from the original notebook).
- Optional local CSV files in `data/`.

## Outputs
- Saved/loaded sweep CSVs in `data/`.
- Theoretical sweep CSVs in `data_analysis/`.


In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Optional dependency: use scipy when available, otherwise fall back to an analytic 1-parameter LS fit.
try:
    from scipy.optimize import curve_fit
    HAVE_SCIPY = True
except Exception:
    curve_fit = None
    HAVE_SCIPY = False


def locate_repo_root() -> Path:
    """Find repository root by looking for known folders."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "notebooks").exists() and ((p / "data").exists() or (p / "data_analysis").exists()):
            return p
    return Path.cwd()


REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)

DATA_DIR = REPO_ROOT / "data"
ANALYSIS_DIR = REPO_ROOT / "data_analysis"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", Path.cwd())
print("Data directory:", DATA_DIR)
print("Analysis directory:", ANALYSIS_DIR)
print("SciPy available:", HAVE_SCIPY)


## Embedded Measurement Data

These arrays are copied from the original `data_analysis.ipynb` so the notebook can run without external lab files.


In [ ]:
# Frequency sweep (GHz)
ACS_frequency_ghz = np.linspace(4.10, 4.35, num=26)

# Measured delta frequencies (Hz)
Q1_delta_freq_array_hz = np.array([
    456358, 440364, 479197, 457785, 382023, 405096, 817702, -2190954,
    -750987, -511354, -427594, -345728, -247638, -183449, -145183,
    -98772, -54644, -35016, -30891, -38599, -40378, -40025, -38624,
    -36081, -29740, -24528
], dtype=float)

Q2_delta_freq_array_hz = np.array([
    -596512, -756523, -1360695, 2114717, 1010825, 671982, 727803, 903658,
    898561, 925287, 1064361, 1004260, 901459, 857606, 967765, 1163793,
    1511051, 3548153, -2950243, -1592068, -1038679, -704298, -502514,
    -389556, -295553, -214057
], dtype=float)

assert len(ACS_frequency_ghz) == len(Q1_delta_freq_array_hz) == len(Q2_delta_freq_array_hz)
print("Loaded points:", len(ACS_frequency_ghz))

# Standalone replacements for cf.* constants used by the original lab notebook.
# These are dataset-consistent values so this notebook remains self-contained.
Q1_freq_hz = 4.167131661442007e9
Q1_ef_freq_hz = Q1_freq_hz - 84.23197492163008e6
Q2_freq_hz = 4.275442110405471e9
Q2_ef_freq_hz = Q2_freq_hz - 154.86240026054388e6
ACS_freq_ref_hz = 4.177e9

print(f"Q1_freq = {Q1_freq_hz/1e9:.6f} GHz, Q1_ef_freq = {Q1_ef_freq_hz/1e9:.6f} GHz")
print(f"Q2_freq = {Q2_freq_hz/1e9:.6f} GHz, Q2_ef_freq = {Q2_ef_freq_hz/1e9:.6f} GHz")


In [ ]:
# Quick scatter check of embedded measurements
plt.figure(figsize=(6.0, 3.8), dpi=150)
plt.scatter(ACS_frequency_ghz, Q1_delta_freq_array_hz * 1e-6, label='Q1 data', color='tab:blue')
plt.scatter(ACS_frequency_ghz, Q2_delta_freq_array_hz * 1e-6, label='Q2 data', color='tab:green', marker='s')
plt.xlabel('ACS Frequency (GHz)')
plt.ylabel('Delta Frequency (MHz)')
plt.title('Embedded AC Stark Frequency Sweep Data')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## Path-Safe Save/Load

This section writes local CSV snapshots and reloads them in a way that does not depend on external absolute paths.


In [ ]:
EXP_NAME = 'ACS_f_sweep'
DATE_TAG = '0408'


def save_csv(path: Path, header: str, arr: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(path, np.asarray(arr, dtype=float), delimiter=',', header=header, comments='')


def load_csv_or_default(path: Path, default_arr: np.ndarray) -> np.ndarray:
    if path.exists():
        loaded = np.loadtxt(path, delimiter=',', skiprows=1)
        loaded = np.atleast_1d(loaded).astype(float)
        print(f"Loaded: {path}")
        return loaded
    print(f"Missing: {path}; using embedded defaults.")
    return np.asarray(default_arr, dtype=float)


file_q1 = DATA_DIR / f'{EXP_NAME}_Q1_delta_freq_array_{DATE_TAG}.csv'
file_q2 = DATA_DIR / f'{EXP_NAME}_Q2_delta_freq_array_{DATE_TAG}.csv'
file_acs = DATA_DIR / f'{EXP_NAME}_ACS_frequency_{DATE_TAG}.csv'

# Save
save_csv(file_q1, 'Q1_delta_freq', Q1_delta_freq_array_hz)
save_csv(file_q2, 'Q2_delta_freq', Q2_delta_freq_array_hz)
save_csv(file_acs, 'ACS_frequency', ACS_frequency_ghz)
print('Saved measurement CSV snapshots.')

# Reload
Q1_delta_loaded_hz = load_csv_or_default(file_q1, Q1_delta_freq_array_hz)
Q2_delta_loaded_hz = load_csv_or_default(file_q2, Q2_delta_freq_array_hz)
ACS_freq_loaded_hz = load_csv_or_default(file_acs, ACS_frequency_ghz) * 1e9

print('Reload lengths:', len(ACS_freq_loaded_hz), len(Q1_delta_loaded_hz), len(Q2_delta_loaded_hz))


## Fitting (Same Algorithm as Original)

Model from the original notebook:
\[
\delta_Q(\omega_d) = \frac{\alpha\,\text{scale}}{2\,\Delta\,(\alpha+\Delta)},\quad
\Delta = \omega_Q - \omega_d,
\quad \alpha = \omega_{Q,ef} - \omega_Q
\]

- Fit only `scale` for Q1 and Q2.
- Then refit Q2 on the filtered range `ACS_freq > 4.13e9`.


In [ ]:
def delta_Q_freq(ACS_freq_hz, Q_freq_hz, Q_ef_freq_hz, scale_factor):
    alpha = Q_ef_freq_hz - Q_freq_hz
    delta_qs = Q_freq_hz - ACS_freq_hz
    return alpha * scale_factor / (2.0 * delta_qs * (alpha + delta_qs))


def fit_scale_factor(ACS_freq_hz, delta_freq_hz, Q_freq_hz, Q_ef_freq_hz, p0):
    """
    Fit one-parameter scale factor with curve_fit when available.
    Fallback: analytic least-squares for a single linear basis.
    """
    if HAVE_SCIPY:
        popt, pcov = curve_fit(
            lambda ACS_freq_hz, scale_factor: delta_Q_freq(ACS_freq_hz, Q_freq_hz, Q_ef_freq_hz, scale_factor),
            ACS_freq_hz,
            delta_freq_hz,
            p0=[p0],
            maxfev=100000,
        )
        return float(popt[0]), pcov

    basis = delta_Q_freq(ACS_freq_hz, Q_freq_hz, Q_ef_freq_hz, 1.0)
    denom = float(np.dot(basis, basis))
    if denom <= 0:
        raise RuntimeError('Fallback LS fit failed: non-positive basis norm.')
    scale = float(np.dot(delta_freq_hz, basis) / denom)
    return scale, np.array([[np.nan]])


# Initial guesses used in the original notebook
scale_factor_Q1 = 1e13
scale_factor_Q2 = 1.5e13

# Full-range fits
scale_factor_Q1, _ = fit_scale_factor(
    ACS_freq_loaded_hz, Q1_delta_loaded_hz, Q1_freq_hz, Q1_ef_freq_hz, p0=scale_factor_Q1
)
scale_factor_Q2, _ = fit_scale_factor(
    ACS_freq_loaded_hz, Q2_delta_loaded_hz, Q2_freq_hz, Q2_ef_freq_hz, p0=scale_factor_Q2
)

# Q2 filtered refit (same condition as original)
filter_condition = ACS_freq_loaded_hz > 4.13e9
ACS_freq_filtered_hz = ACS_freq_loaded_hz[filter_condition]
Q2_delta_filtered_hz = Q2_delta_loaded_hz[filter_condition]

scale_factor_Q2, _ = fit_scale_factor(
    ACS_freq_filtered_hz, Q2_delta_filtered_hz, Q2_freq_hz, Q2_ef_freq_hz, p0=scale_factor_Q2
)

print('scale_factor_Q1 / 1e13 =', scale_factor_Q1 / 1e13)
print('scale_factor_Q2 / 1e13 =', scale_factor_Q2 / 1e13)

# Model predictions
delta_Q1_fit_hz = delta_Q_freq(ACS_freq_loaded_hz, Q1_freq_hz, Q1_ef_freq_hz, scale_factor_Q1)
delta_Q2_fit_hz = delta_Q_freq(ACS_freq_loaded_hz, Q2_freq_hz, Q2_ef_freq_hz, scale_factor_Q2)

# Plots: combined and per-qubit
plt.figure(figsize=(6.0, 3.8), dpi=150)
plt.plot(ACS_freq_loaded_hz * 1e-9, delta_Q1_fit_hz * 1e-6, label='Q1 model', color='tab:blue')
plt.scatter(ACS_freq_loaded_hz * 1e-9, Q1_delta_loaded_hz * 1e-6, label='Q1 data', color='tab:blue', s=16)
plt.plot(ACS_freq_loaded_hz * 1e-9, delta_Q2_fit_hz * 1e-6, label='Q2 model', color='tab:green')
plt.scatter(ACS_freq_loaded_hz * 1e-9, Q2_delta_loaded_hz * 1e-6, label='Q2 data', color='tab:green', marker='s', s=16)
plt.xlabel('ACS Frequency (GHz)')
plt.ylabel('Delta Frequency (MHz)')
plt.title('AC Stark Sweep: Data vs Fitted Model')
plt.grid(True, alpha=0.3)
plt.legend(ncol=2)
plt.show()

plt.figure(figsize=(6.0, 3.3), dpi=150)
plt.plot(ACS_freq_loaded_hz * 1e-9, delta_Q1_fit_hz * 1e-6, label='Q1 model', color='tab:blue')
plt.scatter(ACS_freq_loaded_hz * 1e-9, Q1_delta_loaded_hz * 1e-6, label='Q1 data', color='tab:blue', s=16)
plt.xlabel('ACS Frequency (GHz)')
plt.ylabel('Q1 Delta Frequency (MHz)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(6.0, 3.3), dpi=150)
plt.plot(ACS_freq_loaded_hz * 1e-9, delta_Q2_fit_hz * 1e-6, label='Q2 model', color='tab:green')
plt.scatter(ACS_freq_loaded_hz * 1e-9, Q2_delta_loaded_hz * 1e-6, label='Q2 data', color='tab:green', marker='s', s=16)
plt.xlabel('ACS Frequency (GHz)')
plt.ylabel('Q2 Delta Frequency (MHz)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
import math


def calculate_omega_s(alpha1, alpha2, omega1, omega2, scale):
    part = (
        scale * alpha2 * omega1
        + scale * alpha2 * (alpha1 + omega1)
        + alpha1 * omega2
        + alpha1 * (alpha2 + omega2)
    )
    sqrt_part1 = (
        -scale * alpha2 * omega1
        - scale * alpha2 * (alpha1 + omega1)
        - alpha1 * omega2
        - alpha1 * (alpha2 + omega2)
    ) ** 2
    sqrt_part2 = -4 * (alpha1 + scale * alpha2) * (
        alpha1 * omega2 * (alpha2 + omega2)
        + scale * alpha2 * omega1 * (alpha1 + omega1)
    )
    sqrt_part = math.sqrt(sqrt_part1 + sqrt_part2)

    omega_s1 = (part - sqrt_part) / (2 * (alpha1 + scale * alpha2))
    omega_s2 = (part + sqrt_part) / (2 * (alpha1 + scale * alpha2))
    return omega_s1, omega_s2


alpha1_hz = Q1_ef_freq_hz - Q1_freq_hz
alpha2_hz = Q2_ef_freq_hz - Q2_freq_hz
scale_ratio = scale_factor_Q2 / scale_factor_Q1

omega_s1_hz, omega_s2_hz = calculate_omega_s(alpha1_hz, alpha2_hz, Q1_freq_hz, Q2_freq_hz, scale_ratio)

print(f"omega_s solution 1: {omega_s1_hz:.6e} Hz ({omega_s1_hz * 1e-9:.6f} GHz)")
print(f"omega_s solution 2: {omega_s2_hz:.6e} Hz ({omega_s2_hz * 1e-9:.6f} GHz)")
print(f"ACS_ref - omega_s1: {(ACS_freq_ref_hz - omega_s1_hz) * 1e-6:.3f} MHz")


In [ ]:
# Theoretical dense sweep
freq_start_hz = 4.0e9
freq_end_hz = 4.4e9
freq_step_hz = 0.1e6
ACS_freq_th_hz = np.arange(freq_start_hz, freq_end_hz, freq_step_hz)

delta_Q1_th_hz = delta_Q_freq(ACS_freq_th_hz, Q1_freq_hz, Q1_ef_freq_hz, scale_factor_Q1)
delta_Q2_th_hz = delta_Q_freq(ACS_freq_th_hz, Q2_freq_hz, Q2_ef_freq_hz, scale_factor_Q2)

plt.figure(figsize=(6.3, 4.0), dpi=150)
plt.plot(ACS_freq_th_hz * 1e-9, delta_Q1_th_hz * 1e-6, label='Q1 theory', color='tab:blue')
plt.scatter(ACS_freq_loaded_hz * 1e-9, Q1_delta_loaded_hz * 1e-6, label='Q1 data', color='tab:blue', s=14)
plt.plot(ACS_freq_th_hz * 1e-9, delta_Q2_th_hz * 1e-6, label='Q2 theory', color='tab:green')
plt.scatter(ACS_freq_loaded_hz * 1e-9, Q2_delta_loaded_hz * 1e-6, label='Q2 data', color='tab:green', marker='s', s=14)
plt.xlim(4.10, 4.35)
plt.ylim(-5.0, 5.0)
plt.xlabel('ACS Frequency (GHz)')
plt.ylabel('Delta Frequency (MHz)')
plt.title('Measured vs Theoretical AC Stark Shifts')
plt.grid(True, alpha=0.3)
plt.legend(ncol=2)
plt.show()

print('delta_Q at omega_s1 (Q1):', delta_Q_freq(omega_s1_hz, Q1_freq_hz, Q1_ef_freq_hz, scale_factor_Q1))
print('delta_Q at omega_s1 (Q2):', delta_Q_freq(omega_s1_hz, Q2_freq_hz, Q2_ef_freq_hz, scale_factor_Q2))

# Save theory outputs
file_q1_th = ANALYSIS_DIR / f'{EXP_NAME}_delta_Q1_freq_th_{DATE_TAG}.csv'
file_q2_th = ANALYSIS_DIR / f'{EXP_NAME}_delta_Q2_freq_th_{DATE_TAG}.csv'
file_acs_th = ANALYSIS_DIR / f'{EXP_NAME}_ACS_freq_th_{DATE_TAG}.csv'

save_csv(file_q1_th, 'delta_Q1_freq_th', delta_Q1_th_hz)
save_csv(file_q2_th, 'delta_Q2_freq_th', delta_Q2_th_hz)
save_csv(file_acs_th, 'ACS_freq_th', ACS_freq_th_hz)

print('Saved theory CSVs:')
print(' -', file_q1_th)
print(' -', file_q2_th)
print(' -', file_acs_th)
